# Round 2 Manual — Iteration 3 (completed)

This notebook is the readable companion to the completed iteration-3 analysis. The heavy simulation work is already materialised in `results/iteration3/`, so the notebook stays readable and reproducible.

## 1. What changed vs the first iteration-3 draft?

- kept the exact `Research/Scale` solver
- kept the exact tie-aware rank engine
- fixed the notion of regret to use the **per-simulation best v**
- added sensitivity to **focal-point locations** and **just-above locations**
- added a final markdown summary and explicit final recommendation

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd() if Path.cwd().name == 'manual' else Path('/Users/pablo/Desktop/prosperity/round_2/manual')
OUT = BASE / 'results' / 'iteration3'
PLOTS = OUT / 'plots'

rs = pd.read_csv(OUT / 'optimal_rs_by_speed_iteration3.csv')
pmf = pd.read_csv(OUT / 'mixture_total_pmf.csv')
ev = pd.read_csv(OUT / 'ev_by_speed_iteration3.csv')
sens = pd.read_csv(OUT / 'sensitivity_summary_iteration3.csv')
rec = pd.read_csv(OUT / 'final_recommendation_iteration3.csv')
rec

## 2. Exact Research/Scale subproblem

For each `v`, the code solves the exact integer split `r*(v), s*(v)` first, then feeds the resulting gross value into the Monte Carlo rank game.

![](results/iteration3/plots/00_rs_subproblem.png)

## 3. Exact ranking engine with ties

Rank is computed as `# strictly higher speeds + 1`, so ties share the minimum rank of the tied block. That is the exact interpretation of the manual rules.

The checks are documented in the markdown summary and in the code runner.

## 4. Explicit type distributions and total mixture

The field distribution is not a black box: each type is translated into a PMF, then the total mixture is formed by the user-provided weights.

![](results/iteration3/plots/01_component_pmfs.png)

![](results/iteration3/plots/02_component_overlay_zoom.png)

![](results/iteration3/plots/03_mixture_total.png)

In [ ]:
pmf.sort_values('pmf', ascending=False).head(12)

## 5. Monte Carlo payoff surface

The main Monte Carlo uses many sampled populations of other players. For every candidate `v`, it computes the exact induced rank, multiplier and payoff using the exact `r*(v), s*(v)`.

![](results/iteration3/plots/04_ev_by_speed.png)

![](results/iteration3/plots/05_regret_by_speed.png)

![](results/iteration3/plots/06_rank_multiplier_diagnostics.png)

![](results/iteration3/plots/07_dashboard.png)

In [ ]:
ev.loc[ev['v'].isin([41,42,43,44,45]), ['v','mean_pnl','p10','p90','expected_regret','prob_best_response']]

## 6. Sensitivity

The completed battery now covers:

- total players `N`
- AI cluster version / location
- AI weight
- focal-point weight
- just-above weight
- focal-point locations
- just-above locations

![](results/iteration3/plots/08_sensitivity_families.png)

![](results/iteration3/plots/09_sensitivity_aggregate.png)

In [ ]:
sens[['label','best_v','robust_v','best_ev']].sort_values(['group','best_v'])

## 7. Final recommendation

![](results/iteration3/plots/10_final_recommendation_zoom.png)

Central recommendation: **Speed = 43, Research = 15, Scale = 42**.

- If you want pure central EV: **43**.
- If you want a robust choice under the tested battery: **43** remains the centre.
- If you want a more conservative hedge against a higher-speed field: **46** is the main alternative.

In [ ]:
rec